In [48]:
import os
import re
import json
from datetime import datetime
import random
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

# 1. Test Dataset Generation from Documents

In [ ]:
from evaluation.eval import prepare_testset_documents, generate_qa_dataset, generate_rag_responses

In [ ]:
docs = prepare_testset_documents("eval_data")

In [ ]:
from rag_pipeline.query_engine import vectorstore

In [ ]:
vectorstore_db = vectorstore(persist_directory="/Volumes/LaCie/Projects_portfolio/IntelliQA/chroma_db", 
                                     documents=docs)

# 2. Synthetic QA Dataset Generation

In [49]:
# imports for synthetic QA Dataset Generation
from ragas import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama # ollama can run locally
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
# csv files are temporarily filtered out because they are large and dominate the dataset
gen_docs = [file for file in docs if not file.metadata['filename'].endswith('.csv')]

# subsample documents to control cost and runtime during synthetic QA generation
random_docs = random.sample(gen_docs, 200)

run_config = RunConfig(max_workers=2, timeout=180)

In [ ]:
generator_llm = LangchainLLMWrapper(
                ChatOllama(
                    model="qwen2.5",
                    temperature=0,
                    # model_kwargs={"response_format":{"type":"json_object"}}
                    format="json"
                    )
                )

generator_embeddings = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2"
                    )
                )           

In [ ]:
testset = generate_qa_dataset(
                                random_docs, generator_llm, generator_embeddings,  
                                run_config=run_config,
                                testset_size=100
                            )

In [ ]:
df = testset.to_pandas()

In [ ]:
testset.to_pandas().to_json("test_data/rag_evaluation_testset.json", orient="records", indent=2)

# 3. RAG Pipeline Execution

In [ ]:
rag_results = generate_rag_responses(df[:], vectorstore_db, session_id="test_session")

In [ ]:
print(rag_results[-1]['user_input'])
print(df.iloc[27].user_input)

# 4. RAG Evaluation

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
evaluator_llm = LangchainLLMWrapper(
                    ChatGoogleGenerativeAI(
                        model="gemini-flash-lite-latest",
                        temperature=0,
                        # model_kwargs={"response_format": {"type": "json_object"}},
                    )
                )
evaluator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )

In [ ]:
metrics = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

run_config = RunConfig(max_workers=2, timeout=180)

In [ ]:
result = evaluate(dataset=EvaluationDataset.from_list(rag_results), metrics=metrics, 
                   run_config=run_config, raise_exceptions=True)

In [ ]:
result_df = result.to_pandas()

In [ ]:
result_df.to_csv("evaluation/rag_eval_v1.csv", index=False)